In [ ]:
!pip install transformers sentencepiece gradio

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr
import torch


In [ ]:

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3-3b")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3-3b")


In [ ]:

def summarize_text(text):
    prompt = f"Summarize the following medical text:\n{text}\n\nSummary:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=150)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

def write_article(text):
    prompt = f"Based on the following information, write a research article:\n{text}\n\nArticle:"
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=300)
    article = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return article

def redact_phi(text):
    redacted_text = text.replace("Patient Name", "[REDACTED]")
    return redacted_text


In [ ]:

def multi_agent_system(input_text, task):
    if task == "Summarization":
        return summarize_text(input_text)
    elif task == "Article Writing":
        return write_article(input_text)
    elif task == "PHI Redaction":
        return redact_phi(input_text)
    else:
        return "Invalid task selected."


In [ ]:

with gr.Blocks() as demo:
    gr.Markdown("# 🧠 Medical Text Multi-Agent AI System")
    gr.Markdown("Upload or paste medical text, choose a task, and view the AI's output.")

    text_input = gr.Textbox(lines=10, label="Medical Text Input")
    task_selector = gr.Radio(["Summarization", "Article Writing", "PHI Redaction"], label="Choose Task")
    output_box = gr.Textbox(label="AI Output")

    submit_btn = gr.Button("Run Agent")
    submit_btn.click(fn=multi_agent_system, inputs=[text_input, task_selector], outputs=output_box)

demo.launch()
